# PantheraLAB 🐆 — Prueba real de IA con tu perro

**Prueba independiente:** MegaDetector V6 encuentra animales/personas y SpeciesNet intenta clasificar la especie. No hay eventos inventados.

1. Tocá **Entorno de ejecución → Ejecutar todas**.
2. Elegí una **fotografía propia** del animal cuando Colab la solicite.
3. Verás los cuadros del detector, las sugerencias de SpeciesNet y un informe descargable.

Si tenés opción de **GPU T4**, podés seleccionarla desde *Entorno de ejecución → Cambiar tipo de entorno*. No es indispensable y Google no garantiza recursos gratuitos.

**Importante:** es un ensayo con fotos, NO video en vivo ni conexión con PantheraLIVE. Las imágenes se procesan en tu sesión de Colab; no se envían a Render ni a PantheraLAB. Evitá imágenes de personas.

Referencias oficiales: https://github.com/microsoft/MegaDetector · https://github.com/google/cameratrapai

## 1. Instalar MegaDetector y SpeciesNet
Solo en la sesión de Colab. Puede tardar varios minutos en descargar paquetes.

In [ ]:
%pip -q install PytorchWildlife speciesnet Pillow

## 2. Elegir la foto del perro u otro animal
Se ejecuta una única prueba sobre tu fotografía. Recomendación: JPG, PNG o WEBP de hasta 12 MB.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps
from pathlib import Path
from io import BytesIO
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se seleccionó una fotografía.")
name, payload = next(iter(uploaded.items()))
if len(payload)>12*1024*1024:
    raise ValueError("Fotografía mayor que 12 MB.")
photo = ImageOps.exif_transpose(Image.open(BytesIO(payload))).convert("RGB")
if photo.width*photo.height>25_000_000:
    raise ValueError("Fotografía mayor que 25 megapíxeles.")
folder = Path("/content/pantheralab_prueba_real")
folder.mkdir(parents=True,exist_ok=True)
file_path = folder/"fotograma.jpg"
photo.save(file_path,quality=92)
preview = photo.copy()
preview.thumbnail((820,820))
display(preview)
print("Fotografía preparada:",photo.width,"x",photo.height)

## 3. Detectar realmente con MegaDetector V6
Identifica **animales, personas o vehículos**, pero no especies. Las cajas son predicciones, no verificación científica.

In [ ]:
import torch
from PIL import ImageDraw
from PytorchWildlife.models import detection as pw_detection
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Procesador:",device)
model = pw_detection.MegaDetectorV6(device=device,pretrained=True,version="MDV6-yolov10-c")
prediction = model.single_image_detection(str(file_path),det_conf_thres=0.30)
objects = prediction.get("detections")
names = {0:"animal",1:"person",2:"vehicle"}
detections = []
if objects is not None:
    for xyxy,conf,class_id in zip(objects.xyxy,objects.confidence,objects.class_id):
        detections.append({
            "class_name":names.get(int(class_id),"unknown"),
            "confidence":round(float(conf),4),
            "bbox_xyxy":[int(round(float(v))) for v in xyxy]
        })
annotated=photo.copy()
draw=ImageDraw.Draw(annotated)
for d in detections:
    box=d["bbox_xyxy"]
    draw.rectangle(box,outline="#00db9d",width=4)
    draw.text((box[0]+3,max(0,box[1]-16)),
              d["class_name"].upper()+" "+str(round(d["confidence"]*100))+"%",
              fill="#00db9d")
view=annotated.copy()
view.thumbnail((920,920))
display(view)
annotated.save(folder/"detecciones.jpg",quality=92)
print("Animales:",sum(d["class_name"]=="animal" for d in detections))
print("Personas:",sum(d["class_name"]=="person" for d in detections))
print("Vehículos:",sum(d["class_name"]=="vehicle" for d in detections))

## 4. Sugerir especie con SpeciesNet
Solo si MegaDetector detectó un animal **y no hay personas**. Usamos la clasificación oficial v4.0.3a sobre el recorte del animal más probable. La primera ejecución descarga pesos adicionales.

In [ ]:
from speciesnet.classifier import SpeciesNetClassifier
animals=sorted((x for x in detections if x["class_name"]=="animal"),
               key=lambda x:x["confidence"],reverse=True)
has_people=any(x["class_name"]=="person" for x in detections)
species_suggestions=[]
species_status="not_run"
if has_people:
    species_status="person_detected"
    print("Escena con personas: no se ejecuta la clasificación por privacidad.")
elif not animals:
    species_status="no_animal"
    print("Sin animal detectado: SpeciesNet no se ejecuta.")
else:
    x1,y1,x2,y2=animals[0]["bbox_xyxy"]
    crop=photo.crop((max(0,x1),max(0,y1),min(photo.width,x2),min(photo.height,y2)))
    display(crop)
    try:
        classifier=SpeciesNetClassifier(
            model_name="kaggle:google/speciesnet/pyTorch/v4.0.3a/1",
            device=device
        )
        result=classifier.predict("animal.jpg",classifier.preprocess(crop))
        classes=result.get("classifications",{})
        species_suggestions=[
          {"label":str(label),"score":round(float(score),4)}
          for label,score in zip(classes.get("classes",[]),classes.get("scores",[]))
        ][:5]
        species_status="suggested" if species_suggestions else "no_prediction"
        for species in species_suggestions:
            print(species["label"],":",round(100*species["score"],1),"% (NO verificado)")
    except Exception as exc:
        species_status="error"
        print("SpeciesNet no pudo ejecutarse:",type(exc).__name__,str(exc)[:260])
        print("Se conserva el resultado de MegaDetector.")

## 5. Resultado científico preliminar
Descarga un JSON y la fotografía marcada. No se sincroniza con PantheraLIVE, no hay diagnóstico de caza furtiva ni identificación individual.

In [ ]:
from datetime import datetime,timezone
import json
summary={
 "platform":"PantheraLAB",
 "test":"Colab MegaDetector V6 + SpeciesNet",
 "timestamp":datetime.now(timezone.utc).isoformat(),
 "simulation":False,
 "scientifically_verified":False,
 "source":"authorized_image_uploaded_to_colab",
 "megadetector":{"model":"MDV6-yolov10-c","detections":detections},
 "speciesnet":{"status":species_status,"suggestions":species_suggestions},
 "saved_in_pantheralive":False
}
report_file=folder/"resultado_pantheralab.json"
report_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))
print("Los resultados se guardan únicamente en el entorno temporal de Colab.")

In [ ]:
from google.colab import files
files.download(str(report_file))
files.download(str(folder/"detecciones.jpg"))

**Lo siguiente para PantheraLIVE:** alojar modelos en un proceso de inferencia privado y persistente con recursos adecuados y conectar la API existente. Colab gratuito no mantiene un servicio 24/7 ni garantiza GPU. Nunca publiques un endpoint abierto que procese imágenes sensibles.